# x402 Payments Demo — Pay-per-Request with AlgoPay SDK

This notebook demonstrates **x402 HTTP payments** — the emerging standard for autonomous AI agent payments.

## What is x402?

x402 activates the dormant HTTP `402 Payment Required` status code. When an agent requests a paid resource:

1. Server responds **402** + payment requirements (amount, recipient, network)
2. Client signs a **USDC payment** and retries with `PAYMENT-SIGNATURE` header
3. Server verifies payment on-chain and returns the **resource + tx hash**

No accounts, no API keys, no sessions — just cryptographic proof of payment.

## What this demo covers

- Detection and simulation of x402 endpoints
- Successful payments to **real public x402 paywalls** (GoPlausible ecosystem)
- Guard enforcement (budget caps, domain allowlists)
- Failure scenarios (cap too low, unauthorized domains, invalid paywalls)
- Multiple successive payments with cumulative tracking
- All transactions are **real on Algorand TestNet** — verify at https://lora.algokit.io/testnet/transaction/

## Ecosystem

- x402 on Algorand: https://algorand.co/agentic-commerce/x402
- GoPlausible (reference implementation): https://x402.goplausible.xyz
- Facilitator: https://facilitator.goplausible.xyz

---

**Run all cells sequentially.** The funded wallet from `demo_colab.ipynb` is reused here.

In [ ]:
# Cell 1 — Install dependencies & connect to dashboard
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "algopay-sdk", "httpx"])

import os, json, asyncio, base64
from urllib.request import Request, build_opener, HTTPCookieProcessor
from urllib.error import HTTPError
from http.cookiejar import CookieJar
from decimal import Decimal

# ─── CONFIGURATION ───
DASHBOARD_URL = "https://algopay-sdk-pay-b17a.vercel.app"
DASHBOARD_EMAIL = "md.hatifosmani15@gmail.com"
DASHBOARD_PASSWORD = "Hatif@150503"

WALLET_PRIVATE_KEY_B64 = "DELsQ7/4HPEL/p3sLLFuEvqSW+6NceLhP4aLz9aH6/XquDnSLlJqmdQhaVQat/iQCcfRAB5NcI5roMvE4ZAynw=="
WALLET_ADDRESS = "5K4DTUROKJVJTVBBNFKBVN7YSAE4PUIADZGXBDTLUDF4JYMQGKP57O3P5Q"
WALLET_ID = "x402-demo-wallet"
WALLET_SET_ID = "x402-demo-set"
# ─────────────────────

# Login to dashboard and get API key for telemetry
jar = CookieJar()
opener = build_opener(HTTPCookieProcessor(jar))

def _api(method, path, body=None):
    url = f"{DASHBOARD_URL}{path}"
    data = json.dumps(body).encode() if body else None
    headers = {"Content-Type": "application/json"} if body else {}
    req = Request(url, data=data, headers=headers, method=method)
    try:
        with opener.open(req, timeout=30) as resp:
            raw = resp.read().decode()
            return resp.status, json.loads(raw) if raw else {}
    except HTTPError as e:
        raw = e.read().decode()
        return e.code, json.loads(raw) if raw else {"error": str(e)}

status, resp = _api("POST", "/api/auth/login", {"email": DASHBOARD_EMAIL, "password": DASHBOARD_PASSWORD})
assert status == 200, f"Dashboard login failed ({status}): {resp}"
print(f"Dashboard connected as {DASHBOARD_EMAIL}")

status, key_resp = _api("POST", "/api/api-keys", {"name": "x402 Demo Key"})
API_KEY = key_resp.get("apiKey", "")
assert API_KEY.startswith("sk_"), f"API key creation failed: {key_resp}"
print(f"API key: {API_KEY[:16]}...")

os.environ["ALGOPAY_CONSOLE_URL"] = DASHBOARD_URL
os.environ["ALGOPAY_API_KEY"] = API_KEY
os.environ["ALGOPAY_NETWORK"] = "algorand-testnet"

# Initialize SDK and register wallet
from algopay import AlgoPay
from algopay.core.types import Network, PaymentMethod

client = AlgoPay(network=Network.ALGORAND_TESTNET)

sk_bytes = base64.b64decode(WALLET_PRIVATE_KEY_B64)
client.wallet.repository.register_wallet(
    wallet_set_id=WALLET_SET_ID,
    wallet_id=WALLET_ID,
    address=WALLET_ADDRESS,
    private_key=sk_bytes,
    network_caip2=Network.ALGORAND_TESTNET.to_caip2(),
    name="x402-demo",
)

try:
    client.wallet.opt_in_usdc(WALLET_ID)
except Exception:
    pass

balance = client.wallet.get_usdc_balance(WALLET_ID)
print(f"\nWallet: {WALLET_ADDRESS}")
print(f"USDC Balance: {balance}")
print(f"Telemetry: {'enabled' if client.telemetry.enabled else 'DISABLED'}")
print(f"\nTelemetry streams to: {DASHBOARD_URL}/dashboard/sdk-events")

In [ ]:
# Cell 2 — Detect x402 endpoints & simulate (dry-run)

WEATHER_URL = "https://x402.goplausible.xyz/examples/weather"

print("=" * 60)
print("x402 ENDPOINT DETECTION")
print("=" * 60)

# The SDK auto-detects that URLs route to the x402 adapter
method = client.detect_method(WEATHER_URL)
print(f"\n  URL: {WEATHER_URL}")
print(f"  Detected method: {method}")

# Compare: a plain Algorand address routes to TRANSFER
method2 = client.detect_method("QDAQERDODYHDSSZGM275THOT3JRGZRQ6MSGZ2LOU5CV2P7JMFIBABBXRXM")
print(f"\n  Algorand address -> {method2}")

print("\n" + "=" * 60)
print("x402 SIMULATE (dry-run — no payment executed)")
print("=" * 60)

# Simulate fetches the 402 requirements and checks if payment would succeed
sim = await client.simulate(WALLET_ID, WEATHER_URL, "1.00")
print(f"\n  Would succeed: {sim.would_succeed}")
print(f"  Route: {sim.route}")
print(f"  Reason: {sim.reason or 'Payment would succeed'}")

In [ ]:
# Cell 3 — Successful x402 payment: GoPlausible Weather API
#
# This makes a REAL payment to the GoPlausible x402 weather paywall.
# The server charges ~$0.01 USDC per request on Algorand TestNet.

print("=" * 60)
print("x402 PAYMENT #1: GoPlausible Weather API")
print("=" * 60)
print(f"  URL: {WEATHER_URL}")
print(f"  Max amount: $1.00 (actual charge will be much less)")
print()

result1 = await client.pay(WALLET_ID, WEATHER_URL, "1.00")

print(f"  Success:      {result1.success}")
print(f"  Status:       {result1.status}")
print(f"  Method:       {result1.method}")
print(f"  Amount paid:  ${result1.amount} USDC")
print(f"  TX hash:      {result1.blockchain_tx}")

if result1.resource_data:
    print(f"\n  Resource data (weather forecast):")
    if isinstance(result1.resource_data, dict):
        print(f"  {json.dumps(result1.resource_data, indent=2)[:600]}")
    else:
        print(f"  {str(result1.resource_data)[:600]}")

if result1.blockchain_tx:
    print(f"\n  Verify on-chain:")
    print(f"  https://lora.algokit.io/testnet/transaction/{result1.blockchain_tx}")

# Collect all tx hashes for final summary
tx_hashes = []
if result1.blockchain_tx:
    tx_hashes.append(result1.blockchain_tx)

In [ ]:
# Cell 4 — x402 payment with tight max cap (still succeeds)
#
# The max_amount parameter constrains how much the SDK will pay.
# $0.01 is just enough since the server charges exactly $0.01.

print("=" * 60)
print("x402 PAYMENT #2: Tight Max Cap ($0.01)")
print("=" * 60)
print(f"  Max amount: $0.01 (server charges $0.01 — just enough)")
print()

await asyncio.sleep(5)  # ensure unique tx ID

result2 = await client.pay(WALLET_ID, WEATHER_URL, "0.01")

print(f"  Success:      {result2.success}")
print(f"  Amount paid:  ${result2.amount} USDC")
print(f"  TX hash:      {result2.blockchain_tx}")

if result2.blockchain_tx:
    tx_hashes.append(result2.blockchain_tx)
    print(f"\n  Demonstrates: cap is tight but sufficient — payment goes through.")
    print(f"  https://lora.algokit.io/testnet/transaction/{result2.blockchain_tx}")

In [ ]:
# Cell 5 — x402 payment #3 (another successful call)
#
# Shows the same endpoint can be called repeatedly.
# Each call = new on-chain transaction.

print("=" * 60)
print("x402 PAYMENT #3: Repeated Call")
print("=" * 60)
print()

await asyncio.sleep(5)

result3 = await client.pay(WALLET_ID, WEATHER_URL, "1.00")

print(f"  Success:      {result3.success}")
print(f"  Amount paid:  ${result3.amount} USDC")
print(f"  TX hash:      {result3.blockchain_tx}")

if result3.blockchain_tx:
    tx_hashes.append(result3.blockchain_tx)
    print(f"\n  https://lora.algokit.io/testnet/transaction/{result3.blockchain_tx}")

print(f"\n  Total x402 transactions so far: {len(tx_hashes)}")

In [ ]:
# Cell 6 — x402 FAILURE: max cap too low
#
# If max_amount is below what the server charges, the x402 policy
# filters out the requirement and the payment fails safely.

print("=" * 60)
print("x402 FAILURE: Max Cap Too Low")
print("=" * 60)
print(f"  URL: {WEATHER_URL}")
print(f"  Max amount: $0.0001 (server requires $0.01)")
print()

result_fail1 = await client.pay(WALLET_ID, WEATHER_URL, "0.0001")

print(f"  Success: {result_fail1.success}")
print(f"  Status:  {result_fail1.status}")
print(f"  Error:   {result_fail1.error}")
print()
print("  The SDK refused to pay because the server's price exceeds our cap.")
print("  No transaction was submitted — wallet funds are safe.")

In [ ]:
# Cell 7 — x402 FAILURE: guard blocks unauthorized domain
#
# RecipientGuard can restrict x402 payments to specific domains.
# Here we add a guard that only allows "allowed-api.com".

print("=" * 60)
print("x402 FAILURE: Domain Guard Block")
print("=" * 60)

# Add restrictive domain guard
await client.add_recipient_guard(
    WALLET_ID,
    mode="whitelist",
    domains=["allowed-api.com"],
    name="x402_strict_domain",
)
print("  Guard added: whitelist domains=[\"allowed-api.com\"]")
print(f"  Attempting: {WEATHER_URL}")
print()

result_fail2 = await client.pay(WALLET_ID, WEATHER_URL, "1.00")

print(f"  Success: {result_fail2.success}")
print(f"  Status:  {result_fail2.status}")
print(f"  Error:   {result_fail2.error}")
print()
print("  The guard blocked payment because x402.goplausible.xyz")
print("  is NOT in the domain whitelist. No funds spent.")

# Remove the restrictive guard for remaining cells
await client.guards.remove_guard(WALLET_ID, "x402_strict_domain")
print("\n  (Guard removed for remaining demos)")

In [ ]:
# Cell 8 — x402 on non-402 URL (free resource)
#
# If the target URL returns 200 directly (no paywall),
# the SDK reports success with $0 cost and no tx.

print("=" * 60)
print("x402 ON FREE RESOURCE (no paywall)")
print("=" * 60)

FREE_URL = "https://httpbin.org/get"
print(f"  URL: {FREE_URL}")
print(f"  This URL returns HTTP 200 — no payment required.")
print()

result_free = await client.pay(WALLET_ID, FREE_URL, "1.00")

print(f"  Success:     {result_free.success}")
print(f"  Amount paid: ${result_free.amount}")
print(f"  TX hash:     {result_free.blockchain_tx}")
print(f"  Method:      {result_free.method}")
print()
print("  Result: Resource was free — no blockchain transaction needed.")

In [ ]:
# Cell 9 — x402 on invalid paywall (402 but no valid x402 JSON)
#
# The SDK gracefully handles servers that return 402 but don't
# follow the x402 protocol specification.

print("=" * 60)
print("x402 ON INVALID PAYWALL")
print("=" * 60)

INVALID_URL = "https://httpbin.org/status/402"
print(f"  URL: {INVALID_URL}")
print(f"  Returns HTTP 402 but with no x402 payment requirements.")
print()

result_invalid = await client.pay(WALLET_ID, INVALID_URL, "1.00")

print(f"  Success: {result_invalid.success}")
print(f"  Status:  {result_invalid.status}")
print(f"  Error:   {result_invalid.error}")
print()
print("  The SDK recognizes this is not a valid x402 endpoint.")
print("  No payment attempted — fails gracefully.")

In [ ]:
# Cell 10 — Multiple successive x402 payments
#
# Shows x402 working for repeated pay-per-request calls.
# Each call produces a unique on-chain transaction.

print("=" * 60)
print("MULTIPLE x402 PAYMENTS (3 requests)")
print("=" * 60)
print(f"  Endpoint: {WEATHER_URL}")
print(f"  Each request = 1 real USDC transaction on Algorand TestNet")
print()

for i in range(3):
    await asyncio.sleep(5)  # ensure unique tx IDs
    r = await client.pay(WALLET_ID, WEATHER_URL, "1.00")
    status_str = "PAID" if r.success else f"FAILED ({r.error})"
    print(f"  Request {i+1}: {status_str} | ${r.amount} USDC | tx={r.blockchain_tx or 'none'}")
    if r.blockchain_tx:
        tx_hashes.append(r.blockchain_tx)

print(f"\n  Total x402 transactions this session: {len(tx_hashes)}")

In [ ]:
# Cell 11 — Verify on-chain + flush telemetry

print("=" * 60)
print("ON-CHAIN VERIFICATION")
print("=" * 60)

print(f"\n  Total x402 transactions: {len(tx_hashes)}")
print(f"  Each is a real USDC transfer on Algorand TestNet.\n")

for i, tx in enumerate(tx_hashes, 1):
    print(f"  TX {i}: {tx}")
    print(f"       https://lora.algokit.io/testnet/transaction/{tx}")
    print()

# Flush telemetry to dashboard
print("  Flushing telemetry to dashboard...")
await asyncio.sleep(5)
if hasattr(client.telemetry, 'flush'):
    await client.telemetry.flush()
    await asyncio.sleep(2)

print(f"\n  All x402 events visible at:")
print(f"  {DASHBOARD_URL}/dashboard/sdk-events")

## Results Summary

### Verified Scenarios

| Scenario | Result |
|----------|--------|
| Payment to x402 weather API | Real USDC tx on testnet |
| Tight max cap ($0.01) | Succeeds (server charges exactly $0.01) |
| Repeated calls | Each produces unique tx hash |
| Cap too low ($0.0001) | Fails safely, no tx submitted |
| Domain guard block | Guard prevents payment to unauthorized domain |
| Free resource (200) | Reports success with $0, no tx |
| Invalid paywall | Detects missing x402 spec, fails gracefully |

### Verify Transactions

Every TX hash above can be verified at: https://lora.algokit.io/testnet/transaction/

### Dashboard

All x402 payment events stream to: https://algopay-sdk-pay-b17a.vercel.app/dashboard/sdk-events

### x402 Ecosystem

| Resource | URL |
|----------|-----|
| x402 on Algorand | https://algorand.co/agentic-commerce/x402 |
| GoPlausible (reference impl) | https://x402.goplausible.xyz |
| Facilitator | https://facilitator.goplausible.xyz |
| x402.org (protocol docs) | https://x402.org |
| Algorand TestNet explorer | https://lora.algokit.io/testnet |

### How the SDK handles x402

```
client.pay(wallet_id, url, max_usdc)
  1. detect_method(url) == PaymentMethod.X402
  2. GET url -> HTTP 402 + JSON payment requirements
  3. Filter accepts[] for algorand-testnet USDC
  4. Check max_usdc >= required_amount
  5. Check guards (budget, domain, single-tx)
  6. Sign USDC payment payload with wallet key
  7. GET url + PAYMENT-SIGNATURE header
  8. Server verifies, submits tx on-chain
  9. 200 + resource + PAYMENT-RESPONSE (tx hash)
 10. Emit telemetry event to dashboard
```